# Notebook 03 — Heavy-Hex Dynamic Circuits

To minimize rerouting, we build the Trotter circuits using the hybrid dynamic-circuit protocol:
- **NN pairs**: ancilla-assisted ZZ via mid-circuit measurement +
  feedforward, with each ancilla pinned to its hardware bridge qubit.
- **Long-range pairs**: bare Rzz gates routed by the transpiler with
  SWAPs (same as the unitary approach).

Reference: Simulation of kicked Ising Hamiltonian with dynamic circuits (https://quantum.cloud.ibm.com/docs/en/tutorials/dc-hex-ising#references)

The circuit is constructed directly on the physical qubits of the
coupling map, so the transpiler cannot insert SWAPs for NN interactions.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
from qiskit.transpiler import CouplingMap

from src.model import coupling_pairs, exact_trotter_magnetization
from src.circuits import (
    build_dynamic_trotter_circuit,               # logical (for simulation)
    build_dynamic_trotter_circuit_physical,       # physical (for metrics)
    transpile_dynamic_physical,                   # transpile physical circuit
)
from src.metrics import extract_metrics, save_metrics, load_metrics

In [2]:
N_QUBITS = 6
ALPHAS = [0.5, 1.0, 2.0, 4.0, 8.0]
CUTOFF_FRAC = 0.01  # keep pairs with J >= 1% of J_max
H_FIELD = 1.0
THETA = np.pi / 4
MAX_STEPS = 8
SEED = 42
GEOMETRY = "chain"

heavy_cmap = CouplingMap.from_heavy_hex(5, bidirectional=True)

## 1. Inspect a dynamic circuit

NN pairs use the ancilla protocol:
reset ancilla → CX(i,anc) → CX(j,anc) → Rz(anc) → H(anc) → measure
→ if 1: Z(i), Z(j)

Non-NN pairs appear as bare Rzz gates (routed with SWAPs at transpile).
Each NN pair gets a dedicated ancilla pinned to the hardware bridge.

In [3]:
# Logical version (for inspection)
circ_log = build_dynamic_trotter_circuit(
    N_QUBITS, alpha=1.0, theta=THETA, n_steps=1, measure=True, cutoff_frac=CUTOFF_FRAC, geometry=GEOMETRY, 
)
print(f"Logical dynamic circuit (alpha=1.0, 1 step):")
print(f"  total qubits: {circ_log.num_qubits} "
      f"({N_QUBITS} data + {circ_log.num_qubits - N_QUBITS} ancilla)")
print(f"  ops: {dict(circ_log.count_ops())}")

# Physical version (for transpilation)
circ_phys = build_dynamic_trotter_circuit_physical(
    N_QUBITS, alpha=1.0, theta=THETA, coupling_map=heavy_cmap,
    n_steps=1, measure=True, cutoff_frac=CUTOFF_FRAC, geometry=GEOMETRY, 
)
print(f"\nPhysical dynamic circuit (alpha=1.0, 1 step):")
print(f"  total qubits: {circ_phys.num_qubits} (full coupling map)")
print(f"  data qubits: {circ_phys.metadata['data_physical_qubits']}")
print(f"  bridge qubits: {circ_phys.metadata['bridge_physical_qubits']}")
print(f"  ops: {dict(circ_phys.count_ops())}")

Logical dynamic circuit (alpha=1.0, 1 step):
  total qubits: 11 (6 data + 5 ancilla)
  ops: {'rx': 12, 'measure': 11, 'cx': 10, 'rzz': 10, 'barrier': 7, 'reset': 5, 'rz': 5, 'h': 5, 'if_else': 5}

Physical dynamic circuit (alpha=1.0, 1 step):
  total qubits: 57 (full coupling map)
  data qubits: [38, 42, 41, 45, 46, 47]
  bridge qubits: [26, 6, 28, 11, 12]
  ops: {'rx': 12, 'measure': 11, 'cx': 10, 'rzz': 10, 'barrier': 7, 'reset': 5, 'rz': 5, 'h': 5, 'if_else': 5}


## 2. Transpile and extract metrics

In [4]:
from src.simulate import simulate_mz

SIMULATE_MZ = False   # Set False to skip Aer simulation (faster, uses exact values)
AER_SHOTS = 10000     # Shots for dynamic circuit simulation

all_metrics = []

for alpha in ALPHAS:
    for n_steps in range(MAX_STEPS + 1):
        # Build on physical topology → transpile (only LR gates get routed)
        cd_phys = build_dynamic_trotter_circuit_physical(
            N_QUBITS, alpha, THETA, coupling_map=heavy_cmap,
            n_steps=n_steps, measure=True, cutoff_frac=CUTOFF_FRAC,
        )
        try:
            cdt = transpile_dynamic_physical(cd_phys, heavy_cmap, seed=SEED)
        except Exception as e:
            print(f"Failed: alpha={alpha}, steps={n_steps}: {e}")
            continue

        m = extract_metrics(cdt, "dynamic", N_QUBITS, alpha, n_steps, THETA)
        m.mz_exact = exact_trotter_magnetization(
            N_QUBITS, alpha, H_FIELD, THETA, n_steps, cutoff_frac=CUTOFF_FRAC,
        )

        if SIMULATE_MZ:
            # Simulate on the *logical* dynamic circuit (not the 57-qubit
            # physical one) — much faster and noiseless result is identical.
            cd_log = build_dynamic_trotter_circuit(
                N_QUBITS, alpha, THETA, n_steps, measure=True,
                cutoff_frac=CUTOFF_FRAC,
            )
            m.mz_sim = simulate_mz(cd_log, N_QUBITS, shots=AER_SHOTS)
        else:
            m.mz_sim = m.mz_exact

        all_metrics.append(m)

save_metrics(all_metrics, "../results/dynamic_metrics.json")
print(f"Saved {len(all_metrics)} records")

Saved 45 records


## 3. Three-way comparison

In [5]:
ideal = load_metrics("../results/ideal_metrics.json")
unitary = load_metrics("../results/unitary_metrics.json")

ideal_1 = {m.alpha: m for m in ideal if m.n_steps == 1}
unitary_1 = {m.alpha: m for m in unitary if m.n_steps == 1}
dynamic_1 = {m.alpha: m for m in all_metrics if m.n_steps == 1}

print(f"{'alpha':>6}  {'ideal':>6}  {'unitary':>8}  {'dynamic':>8}  "
      f"{'u/i':>5}  {'d/i':>5}  {'d/u':>5}  {'SWAPs(d)':>9}")
print("-" * 65)
for alpha in ALPHAS:
    i = ideal_1[alpha]
    u = unitary_1[alpha]
    d = dynamic_1[alpha]
    ui = u.two_q_depth / i.two_q_depth if i.two_q_depth > 0 else 0
    di = d.two_q_depth / i.two_q_depth if i.two_q_depth > 0 else 0
    du = d.two_q_depth / u.two_q_depth if u.two_q_depth > 0 else 0
    print(f"{alpha:>6.1f}  {i.two_q_depth:>6}  {u.two_q_depth:>8}  "
          f"{d.two_q_depth:>8}  {ui:>5.2f}  {di:>5.2f}  {du:>5.2f}  "
          f"{d.swap_count:>9}")

 alpha   ideal   unitary   dynamic    u/i    d/i    d/u   SWAPs(d)
-----------------------------------------------------------------
   0.5       5        16        20   3.20   4.00   1.25         22
   1.0       5        16        20   3.20   4.00   1.25         22
   2.0       5        16        20   3.20   4.00   1.25         22
   4.0       5        16        17   3.20   3.40   1.06         17
   8.0       2         4         4   2.00   2.00   1.00          0


## Observations

With the hybrid physical-qubit approach:

1. **At large $\alpha$ (NN-only regime), dynamic circuits match unitary routing.**  When all interactions are nearest-neighbour, the dynamic circuit has zero SWAPs — the entire Rzz layer is implemented through ancilla-assisted ZZ with no routing.

2. **At small $\alpha$ (many long-range pairs), the dynamic approach still has overhead for the LR Rzz gates**, which need SWAP routing just like the unitary variant. 

3. **Topology matching matters.**  Dynamic circuits is not advantageous when the data-ancilla qubit topology cannot match the hardware topology, especially at small n_qubit when pure SWAP works well. 